<a href="https://colab.research.google.com/github/jairaj023/Internship-Practice/blob/main/Day_16Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import libraries

In [2]:
import re
import ast
import numpy as np
import pandas as pd
from collections import Counter

Load your dataset

In [3]:
from google.colab import files
upload = files.upload()

Saving raw_jobs.csv to raw_jobs.csv


In [5]:
df = pd.read_csv("raw_jobs.csv")

In [6]:
print("Dataset Shape:", df.shape)

Dataset Shape: (10000, 9)


In [7]:
print("Columns:")
print(df.columns.tolist())

Columns:
['job_id', 'job_title', 'company', 'location', 'job_description', 'experience', 'education', 'salary', 'job_type']


In [8]:
display(df.head())

,job_id,job_title,company,location,job_description,experience,education,salary,job_type
0,JOB00001,Python Developer,Infosys,Remote,We are looking for a Python Developer to join ...,1-3 years,MCA,₹10-16 LPA,Part-time
1,JOB00002,Data Scientist,Infosys,"Hyderabad, Telangana",We are looking for a Data Scientist to join ou...,0-2 years,MCA,₹12-20 LPA,Remote
2,JOB00003,Data Engineer,IBM,"Bengaluru, Karnataka",We are looking for a Data Engineer to join our...,8-12 years,BCA,₹3-5 LPA,Internship
3,JOB00004,Business Analyst,Google,"Hyderabad, Telangana",We are looking for a Business Analyst to join ...,1-3 years,M.Tech,₹10-16 LPA,Remote
4,JOB00005,Business Analyst,Mphasis,"Delhi, India",We are looking for a Business Analyst to join ...,8-12 years,M.Tech,₹5-8 LPA,Remote


Check the job description column

In [9]:
df['job_description'] = df['job_description'].fillna("").astype(str)

print("Number of job descriptions:", len(df))

Number of job descriptions: 10000


Create the skill dictionary

In [10]:
skill_dictionary = {
    "Python": ["python", "python3", "python programming"],
    "SQL": ["sql", "structured query language"],
    "Java": ["java"],
    "C++": ["c++", "cpp"],
    "JavaScript": ["javascript", "js"],

    "MySQL": ["mysql"],
    "PostgreSQL": ["postgresql", "postgres"],
    "MongoDB": ["mongodb", "mongo db"],
    "Oracle": ["oracle database", "oracle"],

    "AWS": ["aws", "amazon web services"],
    "Azure": ["azure", "microsoft azure"],
    "GCP": ["gcp", "google cloud", "google cloud platform"],

    "Power BI": ["power bi", "powerbi"],
    "Tableau": ["tableau"],
    "Excel": ["excel", "ms excel", "microsoft excel"],

    "Pandas": ["pandas"],
    "NumPy": ["numpy", "num py"],
    "Scikit-learn": ["scikit-learn", "scikit learn", "sklearn"],
    "TensorFlow": ["tensorflow"],
    "PyTorch": ["pytorch"],

    "Machine Learning": ["machine learning", "ml"],
    "Deep Learning": ["deep learning", "dl"],
    "NLP": ["nlp", "natural language processing"],

    "React": ["react", "react.js", "reactjs"],
    "Angular": ["angular"],
    "Node.js": ["node.js", "nodejs", "node js"],
    "Django": ["django"],
    "Flask": ["flask"],

    "Docker": ["docker"],
    "Kubernetes": ["kubernetes", "k8s"],
    "Git": ["git"],
    "GitHub": ["github"]
}

Create reference / actual skill extraction

In [11]:
def extract_reference_skills(text):
    text = str(text).lower()

    found = []

    for skill, aliases in skill_dictionary.items():
        for alias in aliases:
            pattern = r"\b" + re.escape(alias.lower()) + r"\b"

            if re.search(pattern, text):
                found.append(skill)
                break

    return list(set(found))

Create predicted skill extraction

In [12]:
def extract_predicted_skills(text):
    text = str(text).lower()

    predictions = []

    for skill, aliases in skill_dictionary.items():

        canonical = skill.lower()

        pattern = r"\b" + re.escape(canonical) + r"\b"

        if re.search(pattern, text):
            predictions.append(skill)

    return list(set(predictions))

Test on one example

In [13]:
sample_text = """Experience with Python, SQL, AWS and Power BI."""

actual = extract_reference_skills(sample_text)
predicted = extract_predicted_skills(sample_text)

print("Text:", sample_text)
print("Actual:", actual)
print("Predicted:", predicted)

Text: Experience with Python, SQL, AWS and Power BI.
Actual: ['AWS', 'Python', 'SQL', 'Power BI']
Predicted: ['AWS', 'Python', 'SQL', 'Power BI']


Apply to your 10000 records

In [14]:
df["actual_skills"] = df["job_description"].apply(extract_reference_skills)

df["predicted_skills"] = df["job_description"].apply(extract_predicted_skills)

print("Skill extraction completed!")

Skill extraction completed!


Create error categories

In [15]:
def analyze_errors(row):

    actual = set(row["actual_skills"])
    predicted = set(row["predicted_skills"])

    errors = []

    # False Negative
    missing = actual - predicted

    if missing:
        for skill in missing:
            errors.append({"Error_Type": "False Negative", "Skill": skill, "Actual": skill, "Predicted": "Not detected"})

    # False Positive
    extra = predicted - actual

    if extra:
        for skill in extra:
            errors.append({"Error_Type": "False Positive", "Skill": skill, "Actual": "Not present", "Predicted": skill})

    return errors

Generate the error report

In [16]:
error_records = []

for index, row in df.iterrows():

    errors = analyze_errors(row)

    for error in errors:
        error_records.append({
            "Row_ID": index,
            "Job_Title": row["job_title"],
            "Text": row["job_description"],
            "Actual_Skills": row["actual_skills"],
            "Predicted_Skills": row["predicted_skills"],
            "Error_Type": error["Error_Type"],
            "Skill": error["Skill"],
            "Actual": error["Actual"],
            "Predicted": error["Predicted"]
        })

error_df = pd.DataFrame(error_records)

print("Total errors:", len(error_df))

display(error_df.head(20))

Total errors: 499


,Row_ID,Job_Title,Text,Actual_Skills,Predicted_Skills,Error_Type,Skill,Actual,Predicted
0,18,Backend Developer,We are looking for a Backend Developer to join...,"[Node.js, JavaScript, Python, SQL, Java]","[Python, SQL, Node.js, Java]",False Negative,JavaScript,JavaScript,Not detected
1,49,Backend Developer,We are looking for a Backend Developer to join...,"[JavaScript, Python, SQL, Node.js]","[Python, SQL, Node.js]",False Negative,JavaScript,JavaScript,Not detected
2,69,Full Stack Developer,We are looking for a Full Stack Developer to j...,"[React, JavaScript, Node.js]","[React, Node.js]",False Negative,JavaScript,JavaScript,Not detected
3,87,Backend Developer,We are looking for a Backend Developer to join...,"[JavaScript, Java, Node.js]","[Java, Node.js]",False Negative,JavaScript,JavaScript,Not detected
4,113,Backend Developer,We are looking for a Backend Developer to join...,"[JavaScript, Python, SQL, Node.js]","[Python, SQL, Node.js]",False Negative,JavaScript,JavaScript,Not detected
5,138,Full Stack Developer,We are looking for a Full Stack Developer to j...,"[React, JavaScript, Node.js]","[React, Node.js]",False Negative,JavaScript,JavaScript,Not detected
6,167,Backend Developer,We are looking for a Backend Developer to join...,"[Node.js, JavaScript, Python, SQL, Java]","[Python, SQL, Node.js, Java]",False Negative,JavaScript,JavaScript,Not detected
7,211,Backend Developer,We are looking for a Backend Developer to join...,"[Node.js, JavaScript, Python, SQL, Java]","[Python, SQL, Node.js, Java]",False Negative,JavaScript,JavaScript,Not detected
8,229,Backend Developer,We are looking for a Backend Developer to join...,"[JavaScript, Python, Node.js, Java]","[Python, Node.js, Java]",False Negative,JavaScript,JavaScript,Not detected
9,266,Backend Developer,We are looking for a Backend Developer to join...,"[JavaScript, Java, SQL, Node.js]","[Java, SQL, Node.js]",False Negative,JavaScript,JavaScript,Not detected


Detect synonym problems

In [17]:
synonym_mapping = {
    "ml": "Machine Learning",
    "machine learning": "Machine Learning",
    "powerbi": "Power BI",
    "power bi": "Power BI",
    "postgres": "PostgreSQL",
    "postgresql": "PostgreSQL",
    "python3": "Python",
    "python programming": "Python",
    "scikit learn": "Scikit-learn",
    "sklearn": "Scikit-learn"
}

Detect partial entities

In [18]:
def detect_partial_entity(text):

    text = str(text).lower()

    partial_patterns = {"Python 3": r"\bpython\s*3\b",
        "Python programming": r"\bpython\s+programming\b",
        "AWS Cloud": r"\baws\s+cloud\b",
        "Power BI tool": r"\bpower\s+bi\s+tool\b"
    }

    found = []

    for name, pattern in partial_patterns.items():
        if re.search(pattern, text):
            found.append(name)

    return found

In [19]:
df['partial_entities'] = df['job_description'].apply(detect_partial_entity)

Create detailed error categories

In [21]:
detailed_errors = []

for index, row in df.iterrows():

    text = row["job_description"]

    partial = row["partial_entities"]

    for entity in partial:
        detailed_errors.append({
            "Row_ID": index,
            "Job_Title": row["job_title"],
            "Text": text,
            "Actual": entity.split()[0],
            "Predicted": entity,
            "Error_Type": "Partial Entity"
        })

In [22]:
for index, row in df.iterrows():

    actual = set(row["actual_skills"])
    predicted = set(row["predicted_skills"])

    # False Negative
    for skill in actual - predicted:
        detailed_errors.append({
            "Row_ID": index,
            "Job_Title": row["job_title"],
            "Text": row["job_description"],
            "Actual": skill,
            "Predicted": "Not detected",
            "Error_Type": "False Negative"
        })

    # False Positive
    for skill in predicted - actual:
        detailed_errors.append({
            "Row_ID": index,
            "Job_Title": row["job_title"],
            "Text": row["job_description"],
            "Actual": "Not present",
            "Predicted": skill,
            "Error_Type": "False Positive"
        })

Create final error Dataframe

In [23]:
error_report = pd.DataFrame(detailed_errors)

print("Total errors:", len(error_report))

display(error_report.head(20))

Total errors: 499


,Row_ID,Job_Title,Text,Actual,Predicted,Error_Type
0,18,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
1,49,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
2,69,Full Stack Developer,We are looking for a Full Stack Developer to j...,JavaScript,Not detected,False Negative
3,87,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
4,113,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
5,138,Full Stack Developer,We are looking for a Full Stack Developer to j...,JavaScript,Not detected,False Negative
6,167,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
7,211,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
8,229,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative
9,266,Backend Developer,We are looking for a Backend Developer to join...,JavaScript,Not detected,False Negative


Count error types

In [24]:
error_summary = (error_report["Error_Type"].value_counts().reset_index())

error_summary.columns = ["Error_Type", "Count"]

display(error_summary)

,Error_Type,Count
0,False Negative,499


Save the Excel report

In [25]:
with pd.ExcelWriter("Error_Analysis_Report.xlsx", engine="openpyxl") as writer:

    error_report.to_excel(writer, sheet_name="Error Details", index=False)

    error_summary.to_excel(writer, sheet_name="Error Summary", index=False)

    df[["job_id", "job_title", "actual_skills", "predicted_skills"]].to_excel(writer, sheet_name="Predictions", index=False)

print("Excel report created successfully!")

Excel report created successfully!


Download the report

In [26]:
files.download("Error_Analysis_Report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>